In [1]:
import sys
sys.path.append("/home/suvendu/mlbd/project1/CSL7100-Project/code")  # adjust path if needed

from etl import run_etl


In [2]:
from config import print_config
from etl import run_etl

print_config()
run_etl()

🔧 CONFIGURATION
PROJECT_ROOT: /home/suvendu/mlbd/project1/CSL7100-Project
USE_HDFS: False
BASE_PATH: file:///home/suvendu/mlbd/project1/CSL7100-Project
PATHS:
  raw: file:///home/suvendu/mlbd/project1/CSL7100-Project/raw
  parquet: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet
  graph: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet/graph
  checkpoint: file:///home/suvendu/mlbd/project1/CSL7100-Project/checkpoints
  spark_temp: file:///home/suvendu/mlbd/project1/CSL7100-Project/spark-temp
💻 Running in LOCAL mode


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/16 20:45:44 WARN Utils: Your hostname, suvendu2023, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/16 20:45:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 20:45:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


📥 Loading data from: file:///home/suvendu/mlbd/project1/CSL7100-Project/raw
✅ Removed nulls


💾 Saving to: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet


✅ ETL Completed


In [3]:
#Data Validation (Quick sanity check)

In [4]:
from pyspark.sql import SparkSession
from config import PATHS

spark = SparkSession.builder.getOrCreate()

#df = spark.read.parquet(PATHS["parquet"])
df = spark.read.parquet(PATHS["parquet"])


df.show(5)
df.printSchema()
print("Total records:", df.count())

+--------------+----------+-------+--------------+
|    reviewerID|      asin|overall|unixReviewTime|
+--------------+----------+-------+--------------+
|A2FPZJAG0GHUYV|0790785366|    5.0|    1434412800|
|A3G6EELJOWTJ3W|6300987531|    4.0|    1209427200|
|A2FFYQZPUGOX58|0792151712|    5.0|    1347667200|
|A2LZ7NKENSECA1|0790743213|    4.0|    1376784000|
|A28WNLJUKHW6IE|0788821075|    5.0|    1428969600|
+--------------+----------+-------+--------------+
only showing top 5 rows
root
 |-- reviewerID: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- overall: double (nullable = true)
 |-- unixReviewTime: long (nullable = true)

Total records: 170370


In [5]:
print("Total records:", df.count())

Total records: 170370


In [6]:
#Basic Statistics
df.describe().show()

26/04/16 20:46:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 8:==========================================>              (12 + 4) / 16]

+-------+--------------------+--------------------+------------------+--------------------+
|summary|          reviewerID|                asin|           overall|      unixReviewTime|
+-------+--------------------+--------------------+------------------+--------------------+
|  count|              170370|              170370|            170370|              170370|
|   mean|                NULL| 5.496659318759379E9| 4.219170041674004|1.3844335827575278E9|
| stddev|                NULL|2.0492450254625711E9|1.1677102825834684|1.1591700601469441E8|
|    min|A0009988MRFQ3TROTQPI|          0005019281|               1.0|           889833600|
|    max|       AZZXCFBNEWIBQ|          B01HIUL6WU|               5.0|          1538092800|
+-------+--------------------+--------------------+------------------+--------------------+



In [7]:
from filter_5core import run_5core

run_5core()

💻 Running in LOCAL mode
📥 Loading data from: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet
Initial records: 170370

🔁 Iteration 1
Records before filtering: 170370


26/04/16 20:48:09 WARN DAGScheduler: Broadcasting large task binary with size 1143.9 KiB
                                                                                


🔁 Iteration 2
Records before filtering: 77572



🔁 Iteration 3
Records before filtering: 71997



🔁 Iteration 4
Records before filtering: 71504



🔁 Iteration 5
Records before filtering: 71456



🔁 Iteration 6
Records before filtering: 71452



🔁 Iteration 7
Records before filtering: 71452
✅ Converged
💾 Saving filtered data to: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet/5core


✅ 5-core filtering completed


In [8]:
from encode_ids import run_encoding

run_encoding()

💻 Running in LOCAL mode


26/04/16 20:55:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 20:55:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 20:55:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 20:55:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 20:55:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 20:55:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 2

✅ ID Encoding Completed


In [9]:
from etl import create_spark_session

spark = create_spark_session()



df = spark.read.parquet(PATHS["parquet"] + "/encoded")

df.show(5)
df.printSchema()

💻 Running in LOCAL mode
+-------+-------+-------+--------------+
|user_id|item_id|overall|unixReviewTime|
+-------+-------+-------+--------------+
|   3425|    270|    2.0|    1006473600|
|  17672|   7476|    5.0|    1275782400|
|   1122|  11288|    4.0|    1417219200|
|   9368|   4009|    2.0|    1148774400|
|    327|  12531|    5.0|    1513382400|
+-------+-------+-------+--------------+
only showing top 5 rows
root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- overall: double (nullable = true)
 |-- unixReviewTime: long (nullable = true)



In [10]:
import graph_builder
import importlib

importlib.reload(graph_builder)

<module 'graph_builder' from '/home/suvendu/mlbd/project1/CSL7100-Project/code/graph_builder.py'>

In [11]:
from graph_builder import build_graph
from etl import user_based_split
df = spark.read.parquet(PATHS["parquet"] + "/encoded")

train_df, test_df = user_based_split(df)

vertices, edges = build_graph(spark, train_df)

vertices.show(5)
edges.show(5)

🔀 Performing user-based split (corrected)


Train: 68024, Test: 3428
Max user_id: 24985


Vertices: 37906


Edges: 136048
+---+----+
| id|type|
+---+----+
|148|user|
|463|user|
|471|user|
|496|user|
|833|user|
+---+----+
only showing top 5 rows
+---+-----+-------------------+
|src|  dst|             weight|
+---+-----+-------------------+
|148|25691|                0.5|
|148|25632|                0.5|
|463|31396|                0.8|
|463|36266|                0.2|
|471|28303|0.12500000000000003|
+---+-----+-------------------+
only showing top 5 rows


In [12]:
vertices_test, edges_test = build_graph(spark, test_df)
vertices_test.show(5)
edges_test.show(5)

Max user_id: 24952


Vertices: 5019
Edges: 6856
+-----+----+
|   id|type|
+-----+----+
| 1088|user|
| 3175|user|
| 5156|user|
| 7982|user|
|10817|user|
+-----+----+
only showing top 5 rows
+----+-----+------------------+
| src|  dst|            weight|
+----+-----+------------------+
|1088|29198|0.6666666666666666|
|1088|28186|0.3333333333333333|
|3175|35491|               1.0|
|5156|30034|               1.0|
|7982|31018|               1.0|
+----+-----+------------------+
only showing top 5 rows


In [13]:
from graph_builder import run_graph_builder
from pyspark.sql.functions import max as spark_max
max_user_id = train_df.agg(spark_max("user_id")).collect()[0][0]

run_graph_builder(spark, train_df, max_user_id, mode='train')
run_graph_builder(spark, test_df, max_user_id, mode='test')

Max user_id: 24985


26/04/16 20:56:03 WARN CacheManager: Asked to cache already cached data.
26/04/16 20:56:03 WARN CacheManager: Asked to cache already cached data.


Vertices: 37906
Edges: 136048
💾 Saving vertices...


💾 Saving edges...


✅ Graph Construction Completed for train
✅ Graph Construction Completed
Max user_id: 24952


26/04/16 20:56:08 WARN CacheManager: Asked to cache already cached data.
26/04/16 20:56:08 WARN CacheManager: Asked to cache already cached data.


Vertices: 5019
Edges: 6856
💾 Saving vertices...


💾 Saving edges...


✅ Graph Construction Completed for test
✅ Graph Construction Completed


In [14]:
from etl import create_spark_session
from pyspark.sql.functions import col
spark = create_spark_session()

vertices = spark.read.parquet(PATHS["parquet"] + "/graph/vertices/train")
edges = spark.read.parquet(PATHS["parquet"] + "/graph/edges/train")

💻 Running in LOCAL mode


In [15]:
# 1. User exists
source_user = 148   # example from your edges output
vertices.filter(col("id") == source_user).show()

# 2. User is type=user
vertices.filter((col("id") == source_user) & (col("type") == "user")).show()

# 3. User has edges
edges.filter(col("src") == source_user).show()

# Check if weights sum to 1 per user (important for PPR)
from pyspark.sql.functions import sum

edges.groupBy("src").agg(sum("weight")).show(5)

+---+----+
| id|type|
+---+----+
|148|user|
+---+----+

+---+----+
| id|type|
+---+----+
|148|user|
+---+----+

+---+-----+------+
|src|  dst|weight|
+---+-----+------+
|148|25691|   0.5|
|148|25632|   0.5|
+---+-----+------+

+----+------------------+
| src|       sum(weight)|
+----+------------------+
| 587|               1.0|
| 731|               1.0|
| 988|               1.0|
|1276|               1.0|
|1307|1.0000000000000002|
+----+------------------+
only showing top 5 rows


In [16]:
source_user = edges.select("src").distinct().limit(1).collect()[0][0]

In [17]:
from ppr import personalized_pagerank 
#ranks = personalized_pagerank(spark, vertices, edges, source_user)
#ranks.orderBy("rank", ascending=False).show(10)

In [18]:
from ppr import personalized_pagerank,personalized_pagerank_optimized
from pyspark.sql import SparkSession
from config import PATHS

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setCheckpointDir("/tmp/spark-checkpoints")
# Load graph
vertices = spark.read.parquet(PATHS["parquet"] + "/graph/vertices/train")
edges = spark.read.parquet(PATHS["parquet"] + "/graph/edges/train")

# Choose a user
source_user = 10
ranks, top_items = personalized_pagerank_optimized(
    spark,
    vertices,
    edges,
    source_id=source_user,
    alpha=0.15,
    max_iter=20,
    top_n=10
)

# Show results
top_items.show()

'''ranks = personalized_pagerank(
    spark,
    vertices,
    edges,
    source_id=source_user,
    alpha=0.15,
    max_iter=10
)'''

ranks.orderBy("rank", ascending=False).show(10)

🚀 Starting Personalized PageRank
🔄 Making graph bidirectional
🎯 Source user: 10

🔁 Iteration 1


Max diff: 1.6999999999999997

🔁 Iteration 2


Max diff: 1.4449999999999996

🔁 Iteration 3


Max diff: 1.2282499999999996

🔁 Iteration 4


Max diff: 1.0440125

🔁 Iteration 5
Max diff: 0.8874106250000003

🔁 Iteration 6


...

Max diff: 0.1485021724721278

🔁 Iteration 17


Max diff: 0.12622684660130865

🔁 Iteration 18


Max diff: 0.10729281961111234

🔁 Iteration 19


Max diff: 0.09119889666944551

🔁 Iteration 20


Max diff: 0.07751906216902872

🎯 Extracting Top-N recommendations
+-----+--------------------+----+
|   id|                rank|type|
+-----+--------------------+----+
|34146| 0.08172825216360906|item|
|31512| 0.07328053405957281|item|
|31222| 0.04581668552104332|item|
|30128| 0.02222852433230812|item|
|26058| 0.01936453876475299|item|
|36737| 0.01048382722051867|item|
|37085|0.007289384689532968|item|
|37714|0.006451465333443159|item|
|37342|0.006429065476755739|item|
|31946|0.006106185527204865|item|
+-----+--------------------+----+

+-----+--------------------+
|   id|                rank|
+-----+--------------------+
|   10|  0.1965027979043626|
|34146| 0.08172825216360906|
|31512| 0.07328053405957281|
|31222| 0.04581668552104332|
|  239| 0.03513094120238918|
|24213| 0.02446843508053088|
|30128| 0.02222852433230812|
|19486|0.019509251501705904|
|26058| 0.01936453876475299|
|19119| 0.01603273414344937|
+-----+--------------------+
only showing top 10 rows


In [19]:
# Get items already interacted by user
seen_items = edges.filter(edges.src == source_user) \
                  .select("dst") \
                  .distinct()

# Remove them from recommendations
top_items_filtered = top_items.join(
    seen_items,
    top_items.id == seen_items.dst,
    how="left_anti"
)

top_items_filtered.show()

+-----+--------------------+----+
|   id|                rank|type|
+-----+--------------------+----+
|30128| 0.02222852433230812|item|
|36737| 0.01048382722051867|item|
|37085|0.007289384689532968|item|
|37714|0.006451465333443159|item|
|37342|0.006429065476755739|item|
|31946|0.006106185527204865|item|
+-----+--------------------+----+



In [20]:
from pyspark.sql.functions import collect_list, lit

ground_truth = test_df.groupBy("user_id").agg(
    collect_list("item_id").alias("actual_items")
)

users = test_df.select("user_id").distinct().collect()

In [21]:
len(users)

2300

In [22]:
# per user page rakning for test user - found too slow 

In [23]:
#vertices = vertices.cache()
#edges = edges.cache()
from ppr import personalized_pagerank,personalized_pagerank_optimized_test
vertices.count()
edges.count()

all_recommendations = []

for i, row in enumerate(users):
    user_id = row["user_id"]

    print(f"Running PPR for user: {user_id}")

    ranks, top_items = personalized_pagerank_optimized_test(
        spark,
        vertices,
        edges,
        source_id=user_id,
        alpha=0.15,
        max_iter=10,   # reduce iterations
        top_n=10
    )

    recs = top_items.select(
        col("id").alias("item_id"),
        col("rank")
    ).withColumn("user_id", lit(user_id))

    all_recommendations.append(recs)

    # 🔥 VERY IMPORTANT
    ranks.unpersist()
    top_items.unpersist()

    
    

Running PPR for user: 5
🚀 Starting Personalized PageRank


26/04/16 20:57:20 WARN CacheManager: Asked to cache already cached data.


🎯 Source user: 5

🔁 Iteration 1


Max diff: 0.85

🔁 Iteration 2


Max diff: 0.36124999999999996

🔁 Iteration 3


...

Max diff: 0.020297714246100246

🔁 Iteration 8


Max diff: 0.011530773193802343

🔁 Iteration 9


Max diff: 0.007869247984943983

🔁 Iteration 10


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/suvendu/miniconda3/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/suvendu/miniconda3/lib/python3.11/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/suvendu/miniconda3/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
                                                                                

KeyboardInterrupt: 

In [24]:
# not used

In [25]:
from pyspark.sql.functions import col, explode, lit
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.functions import struct, collect_list, map_from_entries
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.types import MapType, DoubleType
from pyspark.sql.functions import udf

In [26]:
#VALID USER FILTER

In [27]:
graph_users = vertices.filter(col("type") == "user").select("id")

valid_users_df = graph_users.join(
    ground_truth.select(col("user_id").alias("id")),
    "id"
)

user_ids = [r["id"] for r in valid_users_df.collect()]
user_ids_set = set(user_ids)

print("Valid users:", len(user_ids))

Valid users: 2300


In [28]:
#🔹 Step 1: Initialize Users

In [29]:
# Use ONLY test users (critical for evaluation)
test_users = ground_truth.select("user_id").distinct()

user_ids = [r["user_id"] for r in test_users.collect()]
user_ids_set = set(user_ids)


In [30]:
#🔹 Step 2: Initialize Sparse Rank Map

In [31]:
def init_rank(node_id):
    if node_id in user_ids_set:
        return {float(node_id): 1.0}
    return {}

init_udf = udf(init_rank, MapType(DoubleType(), DoubleType()))

ranks = vertices.select("id").withColumn("rank_map", init_udf(col("id")))

In [32]:
init_ranks = ranks

In [33]:
#🔹 Step 3: Helper (Explode Map)

In [34]:
def explode_map(df):
    return df.select(
        col("id"),
        explode("rank_map").alias("user_id", "score")
    ).withColumn(
        "user_id", col("user_id").cast("int")
    )

In [35]:
edges = edges.select("src", "dst", "weight").cache()
edges.count()

26/04/16 22:33:27 WARN CacheManager: Asked to cache already cached data.


136048

In [36]:
#🔹 Step 5: Fast-PPR Iterations

In [37]:
alpha = 0.15
max_iter = 12  # 8

for i in range(max_iter):
    print(f"\nIteration {i+1}")

    e = explode_map(ranks).alias("e")
    g = edges.alias("g")

    # -------------------------
    # Propagation
    # -------------------------
    contribs = e.join(
        g,
        col("e.id") == col("g.src")
    ).select(
        col("g.dst").alias("id"),
        col("e.user_id"),
        (col("e.score") * col("g.weight")).alias("score")
    )

    # -------------------------
    # Aggregate
    # -------------------------
    agg = contribs.groupBy("id", "user_id") \
                  .agg(spark_sum("score").alias("score"))

    # -------------------------
    # Damping
    # -------------------------
    agg = agg.withColumn(
        "score",
        (1 - alpha) * col("score")
    )

    # -------------------------
    # ✅ Correct Teleport
    # -------------------------
    teleport = explode_map(init_ranks) \
        .withColumn("score", alpha * col("score"))

    new_ranks = agg.union(teleport)

    # -------------------------
    # Merge
    # -------------------------
    new_ranks = new_ranks.groupBy("id", "user_id") \
                         .agg(spark_sum("score").alias("score"))

    # -------------------------
    # 🔥 PRUNING (SAFE VALUE)
    # -------------------------
    window = Window.partitionBy("id").orderBy(col("score").desc())

    new_ranks = new_ranks.withColumn(
        "rnk", row_number().over(window)
    ).filter(col("rnk") <= 100).drop("rnk")   # ← increased from 20

    # -------------------------
    # Back to map
    # -------------------------
    ranks = new_ranks.groupBy("id").agg(
        map_from_entries(
            collect_list(struct("user_id", "score"))
        ).alias("rank_map")
    )

    # -------------------------
    # Checkpoint
    # -------------------------
    if i % 2 == 0:
        ranks = ranks.checkpoint()

    # -------------------------
    # DEBUG (IMPORTANT)
    # -------------------------
    print("Non-empty nodes:",
          ranks.selectExpr("size(rank_map) > 0").filter("`(size(rank_map) > 0)`").count())


Iteration 1


Non-empty nodes: 8638

Iteration 2


Non-empty nodes: 28500

Iteration 3


Non-empty nodes: 34567

Iteration 4


Non-empty nodes: 37302

Iteration 5


...

Non-empty nodes: 37873

Iteration 10


Non-empty nodes: 37873

Iteration 11


Non-empty nodes: 37873

Iteration 12


[Stage 17130:=================================================>   (16 + 1) / 17]

Non-empty nodes: 37873


In [38]:
#Step 6: Extract Recommendations (PER USER)

In [39]:
exploded = explode_map(ranks)

recommendations = exploded.select(
    col("user_id"),
    col("id").alias("item_id"),
    col("score").alias("rank")
)

In [40]:
#🔹 Step 7: Keep Only Items

In [41]:
recommendations = recommendations.join(
    vertices,
    recommendations.item_id == vertices.id
).filter(
    col("type") == "item"
).select(
    "user_id", "item_id", "rank"
)

In [42]:
#🔹 Step 8: Reverse Item Shift (IMPORTANT)

In [43]:
max_user_id = train_df.agg({"user_id": "max"}).collect()[0][0]

recommendations = recommendations.withColumn(
    "item_id",
    col("item_id") - (max_user_id + 1)
)

In [44]:
#🔹 Step 9: Remove Seen Items

In [45]:
from pyspark.sql.functions import collect_list, array_contains

train_items = train_df.groupBy("user_id").agg(
    collect_list("item_id").alias("train_items")
)

recommendations = recommendations.join(train_items, "user_id", "left")

recommendations = recommendations.filter(
    ~array_contains(col("train_items"), col("item_id"))
).drop("train_items")

In [46]:
#🔹 Step 10: Top-K per User

In [47]:
K = 5

window = Window.partitionBy("user_id").orderBy(col("rank").desc())

recommendations = recommendations.withColumn(
    "rnk", row_number().over(window)
).filter(col("rnk") <= K).drop("rnk")

In [48]:
#🔹 Step 11: Final Debug (MUST PASS)

In [49]:
print("Recommendations:", recommendations.count())

overlap = recommendations.select("user_id").distinct().intersect(
    ground_truth.select("user_id").distinct()
).count()

print("User overlap:", overlap)

26/04/16 22:36:35 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:36:35 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:36:39 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/16 22:36:42 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/16 22:36:44 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/16 22:36:47 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
                                                                                

Recommendations: 11470


26/04/16 22:36:48 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:36:49 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:36:51 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/16 22:36:54 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/16 22:36:56 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/16 22:36:58 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB


User overlap: 2295


In [50]:
#Step 8: FILTER ONLY TEST USERS (VERY IMPORTANT)

In [51]:
from evaluation import evaluate_precision_recall, compute_map

precision, recall = evaluate_precision_recall(
    recommendations,
    ground_truth,
    k=5
)

pred_items_df = recommendations.groupBy("user_id").agg(
    collect_list("item_id").alias("pred_items")
)

map_k = compute_map(pred_items_df, ground_truth, k=5)

print("\n🎯 FINAL RESULTS")
print(f"Precision@5: {precision}")
print(f"Recall@5: {recall}")
print(f"MAP@5: {map_k}")

26/04/16 22:37:00 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:37:00 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:37:03 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/16 22:37:05 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/16 22:37:07 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/16 22:37:09 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
                                                                                


✅ Precision@5: 0.0022657952069716773
✅ Recall@5: 0.01045751633986928


26/04/16 22:37:10 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:37:11 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:37:13 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/16 22:37:15 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/16 22:37:17 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/16 22:37:19 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB


✅ MAP@5: 0.007716049382716049

🎯 FINAL RESULTS
Precision@5: 0.0022657952069716773
Recall@5: 0.01045751633986928
MAP@5: 0.007716049382716049


In [52]:
print("Recommendations count:", recommendations.count())
recommendations.show(5)

26/04/16 22:37:20 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:37:20 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:37:23 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/16 22:37:25 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/16 22:37:27 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/16 22:37:30 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
                                                                                

Recommendations count: 11470


26/04/16 22:37:31 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:37:31 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/16 22:37:33 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/16 22:37:35 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/04/16 22:37:37 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
[Stage 17309:=================================================>   (16 + 1) / 17]

+-------+-------+--------------------+
|user_id|item_id|                rank|
+-------+-------+--------------------+
|     12|   1623|0.014575710067489228|
|     12|   7352|0.012296133164848407|
|     12|   2850|0.005841136015762814|
|     12|   1323|0.005483918194198337|
|     12|    983|0.005317922537968248|
+-------+-------+--------------------+
only showing top 5 rows


26/04/16 22:37:39 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
                                                                                

In [53]:
print("Ground truth count:", ground_truth.count())
ground_truth.show(5)

Ground truth count: 2300


[Stage 17321:===================================>                 (10 + 5) / 15]

+-------+------------+
|user_id|actual_items|
+-------+------------+
|      5|     [12709]|
|     11|[7589, 8074]|
|     12|[10775, 909]|
|     18|      [5765]|
|     21|[3764, 1757]|
+-------+------------+
only showing top 5 rows


In [54]:
from pyspark.sql.functions import col, count
from pyspark.sql.functions import max as spark_max
from pyspark.sql.functions import min as spark_min
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import collect_list
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
import builtins

In [55]:
# 🔹 Step 1: Build Item Features (Popularity + Recency)

In [56]:
# --- Popularity ---
item_popularity = train_df.groupBy("item_id").agg(
    count("*").alias("popularity")
)

max_pop = item_popularity.agg(spark_max("popularity")).collect()[0][0]

item_popularity = item_popularity.withColumn(
    "popularity",
    col("popularity") / max_pop
)

# --- Recency ---
max_time = train_df.agg(spark_max("unixReviewTime")).collect()[0][0]

item_recency = train_df.withColumn(
    "recency",
    col("unixReviewTime") / max_time
).select("item_id", "recency").dropDuplicates(["item_id"])

# --- Combine ---
item_features = item_popularity.join(item_recency, "item_id")

In [57]:
#Normalize

In [58]:
w = Window.partitionBy()

item_features = item_features \
    .withColumn("pop_norm",
        (col("popularity") - spark_min("popularity").over(w)) /
        (spark_max("popularity").over(w) - spark_min("popularity").over(w) + 1e-9)
    ) \
    .withColumn("rec_norm",
        (col("recency") - spark_min("recency").over(w)) /
        (spark_max("recency").over(w) - spark_min("recency").over(w) + 1e-9)
    )

In [59]:
#Step 3: Convert to Vector

In [60]:
assembler = VectorAssembler(
    inputCols=["pop_norm", "rec_norm"],
    outputCol="features"
)

item_features = assembler.transform(item_features).select("item_id", "features")

In [61]:
# Step 4: Build USER PROFILE (Weighted)

In [62]:
def weighted_avg(vectors, ratings):
    if not vectors:
        return None

    vectors = [list(v) for v in vectors]
    total = builtins.sum(ratings)

    dim = len(vectors[0])
    avg = [0.0] * dim

    for v, r in zip(vectors, ratings):
        for i in range(dim):
            avg[i] += v[i] * r

    return Vectors.dense([x / total for x in avg])

weighted_avg_udf = udf(weighted_avg, VectorUDT())

user_profiles = train_df.join(item_features, "item_id") \
    .groupBy("user_id") \
    .agg(
        collect_list("features").alias("feat_list"),
        collect_list("overall").alias("rating_list")
    ) \
    .withColumn("user_vec", weighted_avg_udf("feat_list", "rating_list")) \
    .select("user_id", "user_vec")

In [63]:
#Step 5: DOT PRODUCT (FIXED SIMILARITY)

In [64]:
def dot_product(v1, v2):
    if v1 is None or v2 is None:
        return 0.0

    v1 = list(v1)
    v2 = list(v2)

    return float(builtins.sum(a*b for a,b in zip(v1, v2)))

dot_udf = udf(dot_product, DoubleType())

In [65]:
# Step 6: Compute Content Score

In [66]:
content_scores = recommendations.join(user_profiles, "user_id") \
    .join(item_features, "item_id") \
    .withColumn(
        "content_score",
        dot_udf(col("user_vec"), col("features"))
    ).select("user_id", "item_id", "content_score")

In [67]:
#Step 7: Normalize Scores

In [68]:
w = Window.partitionBy("user_id")

# PPR normalization
ppr_norm = recommendations.withColumn(
    "ppr_norm",
    (col("rank") - spark_min("rank").over(w)) /
    (spark_max("rank").over(w) - spark_min("rank").over(w) + 1e-9)
)

# Content normalization
content_norm = content_scores.withColumn(
    "content_norm",
    (col("content_score") - spark_min("content_score").over(w)) /
    (spark_max("content_score").over(w) - spark_min("content_score").over(w) + 1e-9)
)

In [69]:
# 🔹 Step 8: Combine (Hybrid Score)

In [70]:
hybrid = ppr_norm.join(content_norm, ["user_id", "item_id"])

w1 = 0.7
w2 = 0.3

hybrid = hybrid.withColumn(
    "final_score",
    w1 * col("ppr_norm") + w2 * col("content_norm")
)

In [71]:
# 🔹 Step 9: Top-K Recommendations

In [72]:
from pyspark.sql.functions import row_number

K = 5

window = Window.partitionBy("user_id").orderBy(col("final_score").desc())

hybrid_recommendations = hybrid.withColumn(
    "rnk", row_number().over(window)
).filter(col("rnk") <= K).select(
    "user_id", "item_id", col("final_score").alias("rank")
)

In [73]:
# 🔹 Step 10: Evaluate

In [74]:
from evaluation import evaluate_precision_recall, compute_map
from pyspark.sql.functions import collect_list

precision, recall = evaluate_precision_recall(
    hybrid_recommendations,
    ground_truth,
    k=5
)

pred_items_df = hybrid_recommendations.groupBy("user_id").agg(
    collect_list("item_id").alias("pred_items")
)

map_k = compute_map(pred_items_df, ground_truth, k=5)

print("\n🎯 HYBRID RESULTS (FIXED)")
print("Precision@5:", precision)
print("Recall@5:", recall)
print("MAP@5:", map_k)

26/04/16 22:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 2


✅ Precision@5: 0.002265795206971678
✅ Recall@5: 0.01045751633986928


26/04/16 22:38:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:38:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:38:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:38:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:38:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:38:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 2

✅ MAP@5: 0.007425562817719679

🎯 HYBRID RESULTS (FIXED)
Precision@5: 0.002265795206971678
Recall@5: 0.01045751633986928
MAP@5: 0.007425562817719679


In [75]:
item_features.show(5)

26/04/16 22:39:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 2

+-------+--------------------+
|item_id|            features|
+-------+--------------------+
|    471|[0.15172413777826...|
|  11748|[0.40689655131443...|
|   2866|[0.03448275858596...|
|   7253|[0.01379310343438...|
|   7554|[0.17241379292984...|
+-------+--------------------+
only showing top 5 rows


In [76]:
item_features.select("features").show(5, False)

26/04/16 22:39:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 2

+-----------------------------------------+
|features                                 |
+-----------------------------------------+
|[0.15172413777826396,0.8319209018630146] |
|[0.4068965513144352,0.8379943481754272]  |
|[0.03448275858596908,0.7159604501771852] |
|[0.013793103434387635,0.6271186424909652]|
|[0.17241379292984543,0.8042372861134134] |
+-----------------------------------------+
only showing top 5 rows


In [77]:
content_scores.select("content_score").describe().show()

26/04/16 22:39:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 22:39:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 2

+-------+--------------------+
|summary|       content_score|
+-------+--------------------+
|  count|               11470|
|   mean|  0.5555788988310273|
| stddev| 0.20466113632602057|
|    min|9.104807183974978E-4|
|    max|  1.6648697838156714|
+-------+--------------------+



In [78]:
import graph_builder
import importlib

importlib.reload(graph_builder)

<module 'graph_builder' from '/home/suvendu/mlbd/project1/CSL7100-Project/code/graph_builder.py'>

In [ ]:
# SPARK NODE2VEC (Word2Vec-based)
#Node2Vec:- Learns global graph structure - → captures similarity between users/items
# Random Walks + Word2Vec (Spark ML)
# Nodes appearing in similar walks → similar embeddings

In [79]:
!pip install node2vec --trusted-host pypi.org --trusted-host files.pythonhosted.org

In [80]:
# For Node2Vec (Random Walk + Word2Vec)
# STEP 1: Prepare Adjacency List- Generate Random Walks-

In [99]:
from graph_builder import build_graph_node2vec , generate_walks

# Step 1: Build graph  #  Prepare Adjacency List
adj, max_user_id = build_graph_node2vec(train_df)

# Step 2: Generate walks
walks = generate_walks(adj, num_walks=20, walk_length=25) # 

walks_df = spark.createDataFrame([(w,) for w in walks], ["walk"])

# Step 3: Word2Vec- Train Word2Vec
from pyspark.ml.feature import Word2Vec

word2vec = Word2Vec(
    vectorSize=128,  # increased from 64 
    windowSize=10,   # increased from 5
    minCount=1,
    inputCol="walk",
    outputCol="embedding"
)

model = word2vec.fit(walks_df)

# Step 4: Get Node Embeddings
embeddings = model.getVectors() \
    .withColumn("id", col("word").cast("int")) \
    .select("id", col("vector").alias("embedding"))

Max user_id: 24985


Adjacency nodes: 37906


26/04/17 10:02:54 WARN TaskSetManager: Stage 17693 contains a task of very large size (9153 KiB). The maximum recommended task size is 1000 KiB.
26/04/17 10:03:18 WARN TaskSetManager: Stage 17695 contains a task of very large size (9153 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

In [100]:
# Split Users and Items (Same as your logic)

In [101]:
user_embeddings = embeddings.filter(col("id") <= max_user_id) \
    .withColumnRenamed("id", "user_id")

item_embeddings = embeddings.filter(col("id") > max_user_id) \
    .withColumn(
        "item_id", col("id") - (max_user_id + 1)
    ).select("item_id", "embedding")

In [102]:
# Dot Product Similarity

In [103]:
import builtins
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def dot_product(v1, v2):
    if v1 is None or v2 is None:
        return 0.0
    return float(builtins.sum(a*b for a,b in zip(v1, v2)))

dot_udf = udf(dot_product, DoubleType())

In [104]:
# Candidate Filtering

In [105]:
top_items = train_df.groupBy("item_id").count() \
    .orderBy(col("count").desc()) \
    .limit(8000) \
    .select("item_id")

candidate_items = item_embeddings.join(top_items, "item_id")

In [ ]:
# Generate Recommendations

In [106]:
recs = user_embeddings.alias("u") \
    .crossJoin(candidate_items.alias("i")) \
    .withColumn(
        "score",
        dot_udf(col("u.embedding"), col("i.embedding"))
    ).select(
        col("u.user_id"),
        col("i.item_id"),
        col("score")
    )

In [ ]:
# Top-K per User

In [107]:
K = 5

window = Window.partitionBy("user_id").orderBy(col("score").desc())

recommendations = recs.withColumn(
    "rnk", row_number().over(window)
).filter(col("rnk") <= K).drop("rnk")

In [ ]:
# Evaluate

In [108]:
# Build Ground Truth from test_df

In [109]:
from pyspark.sql.functions import collect_list

ground_truth = test_df.groupBy("user_id").agg(
    collect_list("item_id").alias("actual_items")
)

In [110]:
# Ensure Recommendations Format

In [111]:
recommendations = recommendations.select(
    "user_id", "item_id", col("score").alias("rank")
)

In [94]:
# DEBUG

In [95]:
print("Recommendations count:", recommendations.count())
print("Ground truth count:", ground_truth.count())

common_users = recommendations.select("user_id").distinct() \
    .intersect(ground_truth.select("user_id").distinct())

print("Users in both:", common_users.count())

Recommendations count: 124925


Ground truth count: 2300


26/04/16 23:19:57 WARN TaskMemoryManager: Failed to allocate a page (67108864 bytes), try again.
26/04/16 23:20:19 WARN TaskMemoryManager: Failed to allocate a page (67108864 bytes), try again.
[Stage 17629:=================================================>   (15 + 1) / 16]

Users in both: 2300


In [96]:
# Run Evaluation

In [113]:
recommendations = recommendations.repartition(200)
ground_truth = ground_truth.repartition(200)

recommendations.cache()
ground_truth.cache()

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.executor.memory", "6g")   # increase if possible
spark.conf.set("spark.driver.memory", "4g")

In [112]:
from evaluation import evaluate_precision_recall, compute_map ,evaluate_precision_recall_safe, compute_map_safe
from pyspark.sql.functions import collect_list

precision, recall = evaluate_precision_recall(
    recommendations,
    ground_truth,
    k=5
)

pred_items_df = recommendations.groupBy("user_id").agg(
    collect_list("item_id").alias("pred_items")
)

map_k = compute_map(pred_items_df, ground_truth, k=5)

print("\n🎯 SPARK NODE2VEC RESULTS")
print("Precision@5:", precision)
print("Recall@5:", recall)
print("MAP@5:", map_k)

26/04/17 10:40:28 WARN TaskSetManager: Stage 17704 contains a task of very large size (1738 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):                                (0 + 16) / 16]
  File "/home/suvendu/miniconda3/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/home/suvendu/miniconda3/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe
26/04/17 10:43:06 ERROR Executor: Exception in task 10.0 in stage 17708.0 (TID 117361)
java.lang.OutOfMemoryError: Java heap space
26/04/17 10:43:47 WARN TaskMemoryManager: Failed to allocate a page (8388592 bytes), try again.
26/04/17 10:43:49 WARN TaskMemoryManager: Failed to allocate a page (8388592 bytes), try again.
26/04/17 10:43:50 WARN TaskMemoryManager: Failed to allocate a page (8388592 bytes), try again.
26/04/17 10:43:53 WARN TaskMemoryManager: Failed 

ConnectionRefusedError: [Errno 111] Connection refused

In [98]:
user_profiles.printSchema()
item_features.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- user_vec: vector (nullable = true)

root
 |-- item_id: integer (nullable = true)
 |-- features: vector (nullable = true)

